# Unit Testing Walkthrough

This complete project walkthrough accompanies the current lab. Build a console application and a separate MSTest project, reproduce an intentional logic defect, repair it, and retain the same tests as regression checks. Use VS Code for the project files; individual browser cells are not a substitute for a multi-project test solution.

These explanations and tasks retain the earlier CS1 teaching material.[^restored]


## Prepare the Solution

```{index} Prepare the Solution
```

An application project contains the behavior to test. A test project contains independent examples of expected behavior. A solution groups them without merging their source files. The test project's reference permits it to use the application's public class.

From a terminal in your workspace, run these commands in order:

```bash
mkdir UnitTestProject1
cd UnitTestProject1
dotnet new sln -n UnitTestProject1
dotnet new console -n MathApp
dotnet new mstest -n MathAppTest
dotnet sln add MathApp/MathApp.csproj
dotnet sln add MathAppTest/MathAppTest.csproj
dotnet add MathAppTest/MathAppTest.csproj reference MathApp/MathApp.csproj
```

Run each command from the UnitTestProject1 directory. The solution file extension depends on the SDK default; `dotnet sln` finds it in this directory. There should be two project folders. Do not put both projects' source files into a single project or add a second Main to the generated console project.

If templates need packages, restore with access to the configured NuGet source. SDK/editor installation remains in the Development Tools appendix. The full steps here teach how the two projects collaborate.

## Prepare Application Code

```{index} Prepare Application Code
```

Replace `MathApp/Program.cs` with the following complete file. `BasicMath` must be public because the test project is a different assembly. The `SomeMath` namespace supplies the name imported by the test file. Main is just an application entry point; the tests call BasicMath directly.

**Intentional defect:** Multiply uses `+`. The program builds because both operators are legal for doubles; testing must reveal the incorrect behavior.

```csharp
using System;
namespace SomeMath
{
    internal class Program
    {
        private static void Main(string[] args)
        {
            BasicMath math = new BasicMath();
            Console.WriteLine(math.Multiply(10, 10));
        }
    }
    public class BasicMath
    {
        public double Add(double a, double b) { return a + b; }
        public double Subtract(double a, double b) { return a - b; }
        public double Divide(double a, double b) { return a / b; }
        public double Multiply(double a, double b)
        {
            return a + b; // Intentional bug: tests should expose this.
        }
    }
}
```

Run `dotnet build MathApp/MathApp.csproj`, then `dotnet run --project MathApp/MathApp.csproj`. A successful build checks compilation, not arithmetic intent. Explain why the printed value 20 is wrong for this call.

## Prepare Tests

```{index} Prepare Tests
```

Replace the template test source in MathAppTest with the complete listing below. If the template generated an example test file under another name, remove that example so the intended suite contains exactly these four tests. Keep its generated project file and package references.

The TestClass and TestMethod attributes let the test framework discover tests. Each test creates a fresh BasicMath, calls one method, and checks an independently chosen expected result. `Assert.AreEqual(expected, actual)` belongs to MSTest and runs as part of the tests; it is not a debug-only assertion or an interactive error dialog.

```csharp
using Microsoft.VisualStudio.TestTools.UnitTesting;
using SomeMath;
namespace MathAppTest;
[TestClass]
public class UnitTest1
{
    [TestMethod]
    public void Test_AddMethod()
    {
        BasicMath bm = new BasicMath();
        double result = bm.Add(10, 10);
        Assert.AreEqual(20.0, result);
    }
    [TestMethod]
    public void Test_SubtractMethod()
    {
        BasicMath bm = new BasicMath();
        double result = bm.Subtract(10, 10);
        Assert.AreEqual(0.0, result);
    }
    [TestMethod]
    public void Test_DivideMethod()
    {
        BasicMath bm = new BasicMath();
        double result = bm.Divide(10, 5);
        Assert.AreEqual(2.0, result);
    }
    [TestMethod]
    public void Test_MultiplyMethod()
    {
        BasicMath bm = new BasicMath();
        double result = bm.Multiply(10, 10);
        Assert.AreEqual(100.0, result);
    }
}
```

These integer-valued results are exactly representable as doubles. For calculations involving rounding, use an appropriate tolerance and explain it; do not assume every floating-point computation should match exactly.

## Run, Read, and Repair

```{index} Run, Read, and Repair
```

From the solution directory, run:

```bash
dotnet build
dotnet test MathAppTest/MathAppTest.csproj
```

The significant failure information is:

```text
Failed Test_MultiplyMethod
Assert.AreEqual failed. Expected:<100>. Actual:<20>.
Failed: 1, Passed: 3, Total: 4
```

The runner also prints package/build information, paths, timings, and a stack trace whose exact format varies. The historical transcript reversed expected and actual because its assertion arguments were reversed. Here expected is 100 and actual is 20, matching the source above.

Read the method name and assertion first. The expected value comes from multiplication, not from copying the implementation's answer. Trace the call, find the wrong operator, and change Multiply to `return a * b;`. Run the **same** test command again; all four should pass. Do not weaken or delete the failing test to make the suite green.

Add zero, negative, and fractional cases. Define what Divide should do for a zero denominator before writing a test: C# double division can return infinity or NaN instead of throwing. A different application contract would require explicit validation. Explain what four passing examples establish and what remains untested.

## Run in VS Code

```{index} Run in VS Code
```

Open the solution folder. With the C# testing support configured, build the projects and inspect Testing/Test Explorer. Run the suite, inspect the failed multiplication test before the repair, then rerun after the repair. Command-line testing remains the reproducible check if the editor has not discovered tests yet.

Submit both projects and the solution, the failing and passing test transcripts, the one-line repair, and your added test cases. Exclude generated bin/obj folders. Explain the relationship among public visibility, namespace import, project reference, test discovery, and assertion. The current lab's additional method and testing tasks remain part of its assignment.

```{rubric} Footnotes
```

[^restored]: Restores the complete 2025 unit-testing lab workflow and listings, with current dotnet commands, explicit MSTest imports, and corrected expected/actual assertion order. The deliberate multiplication defect remains clearly labeled.